In [6]:
# Mounting Drive
from google.colab import drive
drive.mount('/content/drive')

import os

os.listdir("/content/drive/My Drive/ECED/Inonest/AI/BERT_Sentiment/")

Mounted at /content/drive


['256_tokenizer', '256_Frozen_Baseline']

In [16]:
# Pipeline

# Importing functionality
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, BertForSequenceClassification

# Loading tokenizer
tokenizer = AutoTokenizer.from_pretrained("/content/drive/My Drive/ECED/Inonest/AI/BERT_Sentiment/512_tokenizer")

# Loading model
model = BertForSequenceClassification.from_pretrained("/content/drive/My Drive/ECED/Inonest/AI/BERT_Sentiment/256_8to11_unfrozen")
# Test
model.config.id2label = {0: "negative", 1: "positive"}
model.config.label2id = {"negative": 0, "positive": 1}

# Pipeline code
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict_sentiment_with_confidence(
    text,
    model,
    tokenizer,
    device,
    max_length=256
):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=1)
        confidence, prediction = torch.max(probs, dim=1)

    return prediction.item(), confidence.item()




Loading weights:   0%|          | 0/201 [00:02<?, ?it/s]

In [21]:
# Actually using pipline
text = input("Please enter example")

pred, conf = predict_sentiment_with_confidence(
    text, model, tokenizer, device
)

print(f"Predicted label: {pred}")
print(f"Confidence: {conf:.3f}")

Please enter exampleThe movies was amazing
Predicted label: 1
Confidence: 0.927
